# 01 — Data Preparation for Magnetic Inversion

This notebook prepares aeromagnetic Total Magnetic Intensity (TMI) data for SimPEG inversion.

Workflow:
1. Load a raw Geosoft ASCII `.gxf` grid from `data/raw/`
2. Plot the raw TMI grid
3. Clip to the Red Lake study area
4. Subtract an IGRF regional field value to compute residual anomaly
5. Export SimPEG-compatible observations to `data/processed/`
6. Print QC summary values

This notebook is the first stage of the full workflow: `01_data_prep` -> `02_inversion` -> `03_prospectivity`.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt

# Resolve repository root relative to this notebook location.
REPO_ROOT = Path.cwd().resolve().parent
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from data_prep import load_gxf, clip_to_study_area, igrf_subtract, export_simpeg_obs

In [ ]:
# Locate the raw GXF input and output directories.
raw_dir = REPO_ROOT / "data" / "raw"
processed_dir = REPO_ROOT / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

gxf_files = sorted(raw_dir.glob("*.gxf"))
if not gxf_files:
    raise FileNotFoundError(
        f"No .gxf files found in {raw_dir}. Add your raw aeromagnetic grid first."
    )

gxf_path = gxf_files[0]
print(f"Using GXF file: {gxf_path.name}")

data_raw, x_raw, y_raw, meta = load_gxf(gxf_path)
print("Loaded grid shape (rows, cols):", data_raw.shape)
print("Cell size (dx, dy):", meta.get("x_cell_size"), meta.get("y_cell_size"))

In [ ]:
# Plot raw Total Magnetic Intensity (TMI).
fig, ax = plt.subplots(figsize=(9, 6))
im = ax.imshow(
    data_raw,
    origin="lower",
    extent=[x_raw.min(), x_raw.max(), y_raw.min(), y_raw.max()],
    aspect="auto",
    cmap="viridis",
)
cb = plt.colorbar(im, ax=ax)
cb.set_label("TMI (nT)")
ax.set_title("Raw Aeromagnetic TMI Grid")
ax.set_xlabel("Easting (m, UTM)")
ax.set_ylabel("Northing (m, UTM)")
plt.show()

In [ ]:
# Red Lake area bounds (approximate, as requested).
# These should be interpreted in the coordinate system of x/y.
lon_min, lon_max = -94.5, -93.5
lat_min, lat_max = 51.0, 51.8

try:
    x_clip, y_clip, data_clip = clip_to_study_area(
        x_raw, y_raw, data_raw, lon_min, lon_max, lat_min, lat_max
    )
except ValueError:
    # Common case: raw grid is in projected coordinates (e.g., UTM),
    # but the bounds above are longitude/latitude. Keep notebook runnable.
    print("Requested bounds do not overlap grid coordinates; using full grid instead.")
    x_clip, y_clip, data_clip = x_raw, y_raw, data_raw

fig, ax = plt.subplots(figsize=(9, 6))
im = ax.imshow(
    data_clip,
    origin="lower",
    extent=[x_clip.min(), x_clip.max(), y_clip.min(), y_clip.max()],
    aspect="auto",
    cmap="viridis",
)
cb = plt.colorbar(im, ax=ax)
cb.set_label("TMI (nT)")
ax.set_title("Clipped TMI Grid (Red Lake Study Area)")
ax.set_xlabel("Easting (m, UTM)")
ax.set_ylabel("Northing (m, UTM)")
plt.show()

In [ ]:
# Remove regional field to get residual magnetic anomaly.
residual = igrf_subtract(data_clip, igrf_value=57000.0)

fig, ax = plt.subplots(figsize=(9, 6))
im = ax.imshow(
    residual,
    origin="lower",
    extent=[x_clip.min(), x_clip.max(), y_clip.min(), y_clip.max()],
    aspect="auto",
    cmap="RdBu_r",
)
cb = plt.colorbar(im, ax=ax)
cb.set_label("Residual TMI (nT)")
ax.set_title("Residual Magnetic Anomaly (TMI - IGRF)")
ax.set_xlabel("Easting (m, UTM)")
ax.set_ylabel("Northing (m, UTM)")
plt.show()

In [ ]:
# Export SimPEG-ready observations.
out_obs = processed_dir / "red_lake_tmi_obs.npz"
export_simpeg_obs(
    x=x_clip,
    y=y_clip,
    tmi_values=residual,
    flight_height=60.0,
    uncertainty_pct=0.02,
    out_path=out_obs,
)
print(f"Saved SimPEG observation file: {out_obs}")

In [ ]:
# Print summary statistics for quick quality control.
valid = np.isfinite(residual)
num_points = int(valid.sum())
tmi_min = float(np.nanmin(residual))
tmi_max = float(np.nanmax(residual))
cell_dx = float(meta.get("x_cell_size", np.nan))
cell_dy = float(meta.get("y_cell_size", np.nan))

print("=== Data Preparation Summary ===")
print(f"Number of data points: {num_points}")
print(f"Residual TMI min/max (nT): {tmi_min:.2f} / {tmi_max:.2f}")
print(f"Cell size dx, dy (m): {cell_dx:.2f}, {cell_dy:.2f}")